In [1]:
import pandas as pd
import geopandas as gpd
from pathlib import Path
from functools import reduce

from parreg.process_config import load_and_validate_config
from parreg import utils, utils_algo, funcs_clust, funcs_dist

import time
import os

from parreg.logging_config import setup_logging
setup_logging()

import logging
logger = logging.getLogger(__name__)

In [2]:
# read and validate config
config_file = Path('/home/yuqiong.liu/work/Gitlab/ngen-regionalization/configs/config.yaml')
if not config_file.exists():
    raise FileNotFoundError(config_file)

config = load_and_validate_config(config_file)

2025-05-12 18:21:04,555 - parreg.process_config - INFO - Saving config to /home/yuqiong.liu/work/data/ngen_reg/outputs/run_ngen_hlr/config_final.yaml


In [3]:
# process by VPU
vpu = config.general.vpu_list[0]

In [4]:
# Check donors and receivers
donors_dict = config.donor.get_qualified_donors(vpu)

gdf = gpd.read_file(config.general.hydrofabric_file[vpu], layer = 'divides')
gdf_donors = gdf[gdf['divide_id'].isin(donors_dict['divide_id'])]
gdf_receivers = gdf[~gdf['divide_id'].isin(donors_dict['divide_id'])]
donors = gdf_donors['divide_id'].tolist()
receivers = gdf_receivers['divide_id'].tolist()
logger.info(f"Number of donors in vpu {vpu}: {len(donors)}")
logger.info(f"Number of receivers in vpu {vpu}: {len(receivers)}")

2025-05-12 18:21:04,661 - parreg.config_schema - INFO - Initial donors based on all gages in /home/yuqiong.liu/work/data/ngen_reg/gages_nwm4_calib_all.csv
2025-05-12 18:21:04,691 - parreg.config_schema - INFO - Number of initial donors for vpu 01: 88 gages, 4559 divides
2025-05-12 18:21:04,717 - parreg.config_schema - INFO - Number of donors after filtering: 77 gages, 3649 divides
2025-05-12 18:21:04,955 - __main__ - INFO - Number of donors in vpu 01: 3649
2025-05-12 18:21:04,956 - __main__ - INFO - Number of receivers in vpu 01: 16918


In [5]:
# compute the donor-receiver spatial distance
out = config.output.spatial_distance
dist_file = Path(out.path, 'donor_receiver_dist_' + config.general.domain + '_vpu' + vpu + '.' + out.format)
if dist_file.exists():
    logger.info(f"Spatial distance file already exists: {dist_file}\nSkip computing.")
    df_spatial_dist = utils.read_table(dist_file)
    #TODO: check if the spatial distance data includes all pairs of donors and receivers
    # if not, identify the missing pairs and compute the distance for them
else:
    logger.info(f'compute donor-receiver spatial distance ...')
    start_time = time.time()
    df_spatial_dist = utils_algo.compute_pairwise_centroid_distances(gdf_donors, gdf_receivers, 'divide_id', 'divide_id')
    end_time = time.time()
    print(f"Execution time: {end_time - start_time:.4f} seconds")

    # save the spatial distance data
    if out.save:
        if not dist_file.parent.is_dir():
            dist_file.parent.mkdir(parents=True, exist_ok=True)
        
        utils.save_data(df_spatial_dist, dist_file, index=True)
        logger.info(f"Spatial distance data saved to {dist_file}")  

2025-05-12 18:21:04,964 - __main__ - INFO - Spatial distance file already exists: /home/yuqiong.liu/work/data/ngen_reg/outputs/run_ngen_hlr/spatial_distance/donor_receiver_dist_conus_vpu01.parquet
Skip computing.


In [6]:
# process attribute data
datasets = config.general.attr_dataset_list
logger.info(f"Processing attribute data for VPU {vpu} ... datasets: {datasets}")

df_attrs_all = []
for dataset_name in datasets:

    dataset = getattr(config.attr_datasets, dataset_name)
    df_attrs = dataset.get_attr_data()
    
    df_attrs = df_attrs.rename(columns=lambda x: x if x == "divide_id" else f"{dataset_name}_{x}")

    # subset the attribute data to only include donors and receivers for the current VPU
    #TODO: add functionality to add additional donors from neighboring VPUs
    df_attrs = df_attrs[df_attrs['divide_id'].isin(donors + receivers)]

    df_attrs_all.append(df_attrs)

# Merge all attribute data frames column-wise, based on divide_id
df_attrs_all= reduce(
    lambda left, right: pd.merge(left, right, on='divide_id', how='outer'),
    df_attrs_all
)

# add a column to indicate whether the divide_id is a donor or receiver
df_attrs_all['is_donor'] = df_attrs_all['divide_id'].isin(donors)
# move the is_donor column to be the second column
df_attrs_all = df_attrs_all[['divide_id', 'is_donor'] + [col for col in df_attrs_all.columns if col not in ['divide_id', 'is_donor']]]

# check if all donors have attribute data
if not set(donors).issubset(df_attrs_all['divide_id']):
    logger.warning(f"Not all donors are included in the attribute data for VPU {vpu}.")
    missing_donors = [x for x in donors if x not in df_attrs_all['divide_id'].values]
    print("Missing donors:")
    print(missing_donors)

# check if all receivers have attribute data
if not set(receivers).issubset(df_attrs_all['divide_id']):
    logger.warning(f"Not all receivers are included in the attribute data for VPU {vpu}.")
    missing_receivers = [x for x in receivers if x not in df_attrs_all['divide_id'].values]
    print("Missing receivers:")
    print(missing_receivers)

# reset donor and receiver lists based on the attribute data
donors = df_attrs_all[df_attrs_all['is_donor']]['divide_id'].tolist()
receivers = df_attrs_all[~df_attrs_all['is_donor']]['divide_id'].tolist()

print(f'Number of donors in attribute data: {len(donors)}')
print(f'Number of receivers in attribute data: {len(receivers)}')

# check if all donors in attribute data are inlcuded in the donor_id of the spatial distance data
if not set(donors).issubset(df_spatial_dist.columns):
    logger.warning(f"Not all donor_ids in the attribute data are present in the spatial distance data for VPU {vpu}.")
    missing_donor_ids = [x for x in donors if x not in df_spatial_dist.columns]
    print("Missing donor_ids:")
    print(missing_donor_ids)

# check if all receivers in attribute data are inlcuded in the receiver_id of the spatial distance data
if not set(receivers).issubset(df_spatial_dist.index):
    logger.warning(f"Not all receiver_ids in the attribute data are present in the spatial distance data for VPU {vpu}.")
    missing_receiver_ids = [x for x in receivers if x not in df_spatial_dist.index]
    print("Missing receiver_ids:")
    print(missing_receiver_ids)

# sort the attribute data by is_donor and divide_id
df_attrs_all = df_attrs_all.sort_values(by=['is_donor', 'divide_id'], ascending=[False, True])

# check percentage of missing data
df_missing = df_attrs_all.isna().mean()*100
if df_missing.sum()>0:
    logger.warning(f"There are missing data for attributes in vpu {vpu}")
    print("Missing data percentage for each attribute:")
    print(df_missing.loc[df_missing > 0])

# save the attribute data
out1 = config.output.attr_data_final
if out1.save:
    if not Path(out1.path).is_dir():
        Path(out1.path).mkdir(parents=True, exist_ok=True)
    out_file = Path(out1.path, 'attr_' + config.general.domain + '_vpu' + vpu + '.' + out1.format)

    logger.info(f"Saving attribute data to {out_file}")
    utils.save_data(df_attrs_all, out_file)

2025-05-12 18:21:05,342 - __main__ - INFO - Processing attribute data for VPU 01 ... datasets: ['ngen', 'hlr']
2025-05-12 18:21:06,570 - __main__ - WARNING - There are missing data for attributes in vpu 01
2025-05-12 18:21:06,572 - __main__ - INFO - Saving attribute data to /home/yuqiong.liu/work/data/ngen_reg/outputs/run_ngen_hlr/attr_data_final/attr_conus_vpu01.parquet


Number of donors in attribute data: 3649
Number of receivers in attribute data: 16918
Missing data percentage for each attribute:
ngen_dksat       0.213935
ngen_psisat      0.213935
hlr_AQPERMNEW    1.808723
hlr_TAVE         1.808723
hlr_PPT          1.808723
hlr_PET          1.808723
hlr_PMPE         1.808723
hlr_SAND         1.808723
dtype: float64


In [7]:
# detemine whether the catchment is snowy (as snowy and non-snowy catchments are processed separately)
if 'snow_frac' in [col.lower() for col in df_attrs_all.columns]:
    # check if the snow_frac column is present in the attribute data
    n1 = df_attrs_all['snow_frac'].isna().sum()
    if n1 > 0:
        logger.warning(f"There are {n1} missing values in the snow_frac column for VPU {vpu}. Setting them to non-snowy.")
        df_attrs_all['snow_frac'] = df_attrs_all['snow_frac'].fillna(0.0)
    
    # create a new column to indicate whether the catchment is snowy
    df_attrs_all['snowy'] = df_attrs_all['snow_frac'].apply(lambda x: True if x >= config.algorithms.general.min_snow_frac else False)

else:
    # if the snow_frac column is not present, set all catchments to non-snowy
    df_attrs_all['snowy'] = False
    logger.warning(f"The snow_frac column is not present in the attribute data for VPU {vpu}. All catchments are set to non-snowy.")


2025-05-12 18:21:06,616 - __main__ - WARNING - The snow_frac column is not present in the attribute data for VPU 01. All catchments are set to non-snowy.


In [8]:
# pairing/regionalization algorithms
functions = {
             'proximity': funcs_dist,
             'gower': funcs_dist,
             'urf': funcs_dist,
             'kmeans': funcs_clust,
             'kmedoids': funcs_clust,
             'hdbscan': funcs_clust,
             'birch': funcs_clust,
             }
funcs = functions.keys()

# run only those methods specified to run in the config file
funcs = [x for x in funcs if x in config.general.algorithm_list]

print(f"Algorithms to run: {funcs}")

Algorithms to run: ['kmeans']


In [ ]:
# loop through regionalization algorithms and scenarios to generate donor-receiver pairings for each algorithm/scenario combination

for func1 in funcs:
    file_name_str = 'pairs_' + func1 + '_' + config.general.domain + '_vpu' + vpu
    outfile = config.output.pairs.get_file_path(file_name_str)
    
    if outfile.exists():
        logger.info(f'Pair file already exist: {outfile}')
        logger.info(f'Skip the current run: {func1}')
        continue

    logger.info(f'\n======== Identify donors using: {func1}\n') 
    df_donor_all  = pd.DataFrame()   
    start_time = time.time()
    config1 = config.model_dump()['algorithms'][func1]
    config1['njobs'] = config.model_dump()['general']['n_procs']
    config1['non_attr_cols'] = ['divide_id', 'is_donor', 'snowy']
    config1['attrs'] = {'main': [x for x in df_attrs_all.columns if x not in config1['non_attr_cols']],
                        'base': ['ngen_elevation','ngen_slope', 'ngen_aspect']}
    df_donor_all = functions[func1].func(config1, df_attrs_all, df_spatial_dist, func1)  
    end_time = time.time()
    logger.info(f"Execution time: {end_time - start_time:.4f} seconds")
    
    # save donor receiver pairing to csv file    
    config.output.pairs.save_data(df_donor_all, outfile)

2025-05-12 18:21:06,638 - __main__ - INFO - 
======== Identify donors using: kmeans



perform clustering using kmeans approach ...

 Total number of receivers to be paired with donors: 16918

------------------------ Round 1--------------------
Excluding 6 attributes: hlr_AQPERMNEW,hlr_TAVE,hlr_PPT,hlr_PET,hlr_PMPE,hlr_SAND
Number of PCs selected: 9
PCA total portion of variance explained ... 0.8347762506304938

======= 16876 snowy basins ========
===== iter1=1 ======
Using 1 processors ...
===== iter1=2 ======
Using 2 processors ...

---------------- iteration = 1 -------------
Number of receivers with donors identified: 0
===== iter1=3 ======
Using 3 processors ...

---------------- iteration = 2 -------------
Number of receivers with donors identified: 0
===== iter1=4 ======
Using 3 processors ...

---------------- iteration = 3 -------------
Number of receivers with donors identified: 0
===== iter1=5 ======
Using 3 processors ...

---------------- iteration = 4 -------------
Number of receivers with donors identified: 0
===== iter1=6 ======
Using 3 processors ...

-

In [ ]:
print(df_donor_all)